# Import Packages

In [1]:
import pandas as pd
import numpy as np

from xgboost import XGBRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Import Data

In [2]:
df = pd.read_csv("data_land_23-25_imputed.csv")
df['tanggal'] = pd.to_datetime(df['tanggal'])

df.head()

,tanggal,stasiun,pm25,pm10,so2,co,o3,no2,max,critical,...,t2m_c,d2m_c,skt_c,sp_hpa,u10,v10,wind_speed,sshf,ssrd,tp_mm
0,2023-01-01,DKI1 (Bunderan HI),55.0,44.0,47.0,10.0,24.0,9.0,55.0,PM25,...,25.556402,22.932364,25.707747,1007.17020,3.359575,0.245406,3.497011,-11828434.0,16794192.0,21.383282
1,2023-01-02,DKI1 (Bunderan HI),43.0,32.0,52.0,9.0,24.0,8.0,52.0,SO2,...,25.909555,22.809381,26.312996,1007.04736,3.690684,0.154770,3.842652,-19416040.0,19450636.0,5.191999
2,2023-01-03,DKI1 (Bunderan HI),35.0,31.0,49.0,9.0,12.0,7.0,49.0,SO2,...,25.244545,22.703054,25.309061,1006.91550,3.759351,1.106680,3.986183,-8033309.5,14011936.0,20.043148
3,2023-01-04,DKI1 (Bunderan HI),47.0,30.0,53.0,11.0,15.0,9.0,53.0,SO2,...,25.186163,22.388178,24.980600,1006.34700,3.171578,0.234373,3.352136,-2020401.0,25899534.0,10.425209
4,2023-01-05,DKI1 (Bunderan HI),50.0,38.0,50.0,13.0,26.0,11.0,50.0,PM25,...,26.122606,22.259958,26.653069,1006.44745,2.284141,0.517420,2.527324,-16885396.0,19027820.0,1.566000


# Train Test Split

In [3]:
features = [
    # # Air Pollutants
    # 'pm10',
    # 'so2',
    # 'co',
    # 'o3',
    # 'no2',

    # Spatial (Coordinates)
    'lat',
    'lon',

    # Temporal 
    'month',
    'day_of_week',
    'is_weekend',

    # Meteorological 
    't2m_c',
    'd2m_c',
    'skt_c',
    'sp_hpa',
    'u10',
    'v10',
    'wind_speed',
    'sshf',
    'ssrd',
    'tp_mm',

    # Lag Feature
    'pm25_lag1'
]

target = "pm25"

In [4]:
split_date = '2024-10-01'

train_df = df[df['tanggal'] < split_date].copy()
test_df  = df[df['tanggal'] >= split_date].copy()

X_train = train_df[features]
y_train = train_df[target]

X_test = test_df[features]
y_test = test_df[target]

In [5]:
print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

print("\nTrain date range:")
print(train_df['tanggal'].min(), "to", train_df['tanggal'].max())

print("\nTest date range:")
print(test_df['tanggal'].min(), "to", test_df['tanggal'].max())

Train shape: (3195, 16)
Test shape : (755, 16)

Train date range:
2023-01-01 00:00:00 to 2024-09-30 00:00:00

Test date range:
2024-10-01 00:00:00 to 2025-02-28 00:00:00


## Train-Test Split

- Data tahun 2023–2024 digunakan sebagai data training karena periode tersebut menyediakan pola historis yang cukup panjang untuk mempelajari karakteristik PM2.5, baik dari sisi temporal maupun pengaruh meteorologis.

- Penggunaan dua tahun data training membantu model mengenali:
  - pola harian,
  - pola musiman,
  - tren jangka pendek,
  - serta fluktuasi PM2.5 pada berbagai kondisi cuaca.

- Data awal tahun 2025 digunakan sebagai data testing untuk mensimulasikan proses prediksi pada data masa depan (*out-of-sample prediction*).

- Pembagian berdasarkan waktu dilakukan untuk menjaga struktur temporal data dan menghindari *data leakage*, karena pada data deret waktu informasi masa depan tidak boleh digunakan dalam proses pelatihan model.

- Pendekatan ini juga lebih realistis dibanding *random split*, karena model dievaluasi berdasarkan kemampuannya memprediksi observasi yang benar-benar belum pernah dilihat sebelumnya.

- Selain itu, penggunaan awal tahun 2025 sebagai data testing memungkinkan evaluasi performa model pada kondisi terbaru sehingga hasil pengujian lebih representatif terhadap implementasi prediksi PM2.5 secara aktual.

# Modelling

In [6]:
xgb_model = XGBRegressor(
    objective='reg:squarederror',

    n_estimators=150,
    learning_rate=0.03,

    max_depth=2,

    min_child_weight=10,

    subsample=0.6,
    colsample_bytree=0.6,

    gamma=1,

    reg_alpha=2,
    reg_lambda=5,

    random_state=1963
)

xgb_model.fit(
    X_train,
    y_train
)

train_pred = xgb_model.predict(X_train)
test_pred  = xgb_model.predict(X_test)

## Arsitektur dan Hyperparameter XGBoost

### Objective Function

- `objective = 'reg:squarederror'`
  - Digunakan karena target PM2.5 merupakan data kontinu sehingga permasalahan dimodelkan sebagai regresi.
  - Objective ini meminimalkan *squared error* antara nilai aktual dan prediksi model.

---

### Jumlah Pohon dan Learning Rate

- `n_estimators = 150`
  - Menentukan jumlah pohon (*trees*) yang dibangun secara bertahap.
  - Nilai 150 dipilih agar model memiliki kapasitas belajar yang cukup tanpa terlalu kompleks.

- `learning_rate = 0.03`
  - Mengatur seberapa besar kontribusi setiap pohon terhadap model akhir.
  - Nilai kecil dipilih agar proses pembelajaran berlangsung lebih stabil dan bertahap.
  - Kombinasi *learning rate* kecil dengan jumlah pohon yang cukup membantu mengurangi risiko *overfitting*.

---

### Kompleksitas Pohon

- `max_depth = 2`
  - Membatasi kedalaman pohon agar model tetap sederhana.
  - Kedalaman rendah membantu model menangkap pola umum tanpa terlalu mengikuti noise pada data PM2.5.

- `min_child_weight = 10`
  - Mengontrol jumlah minimum observasi pada node sebelum dilakukan pemisahan.
  - Nilai yang lebih besar membuat model lebih konservatif sehingga split hanya dilakukan jika benar-benar memberikan informasi penting.

---

### Random Subsampling

- `subsample = 0.6`
  - Hanya 60% data digunakan pada setiap iterasi pembentukan pohon.
  - Teknik ini membantu meningkatkan generalisasi model dan mengurangi *overfitting*.

- `colsample_bytree = 0.6`
  - Hanya sebagian fitur digunakan pada setiap pohon.
  - Membantu mengurangi dominasi fitur tertentu dan meningkatkan keragaman antar pohon.

---

### Regularisasi dan Kontrol Split

- `gamma = 1`
  - Split baru hanya dilakukan jika memberikan penurunan loss yang cukup besar.
  - Membantu menghindari pembentukan cabang yang tidak penting.

- `reg_alpha = 2`
  - Regularisasi L1 untuk mengurangi kompleksitas model.
  - Membantu seleksi fitur secara tidak langsung dengan menekan kontribusi fitur yang kurang penting.

- `reg_lambda = 5`
  - Regularisasi L2 untuk menstabilkan bobot model.
  - Membantu mengurangi variansi dan meningkatkan kemampuan generalisasi.

---

### Reproducibility

- `random_state = 42`
  - Digunakan agar hasil pelatihan model dapat direproduksi secara konsisten pada setiap proses running.

# Evaluation

In [7]:
def regression_metrics(y_true, y_pred):

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)

    return pd.Series({
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    })

In [8]:
train_metrics = regression_metrics(y_train, train_pred)
test_metrics  = regression_metrics(y_test, test_pred)

metrics_df = pd.DataFrame({
    'Train': train_metrics,
    'Test': test_metrics
})

print(metrics_df)

          Train       Test
RMSE  13.710596  16.606425
MAE    9.887546  13.116131
R2     0.627552   0.514456


## Evaluasi Kinerja Model XGBoost

### Kesimpulan Umum

- Model XGBoost menunjukkan performa yang cukup baik pada data training dan masih mempertahankan kemampuan prediksi yang moderat pada data testing.

- Selisih performa antara training dan testing menunjukkan adanya indikasi *overfitting ringan*, tetapi belum terlalu ekstrem karena error testing masih relatif terkendali.

- Hasil ini mengindikasikan bahwa fitur temporal dan meteorologis berhasil membantu model dalam menangkap dinamika PM2.5, meskipun masih terdapat variasi spasial dan faktor lain yang belum sepenuhnya terakomodasi.

In [9]:
importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': xgb_model.feature_importances_
})

importance_df = (
    importance_df
    .sort_values('importance', ascending=False)
)

print("\nFeature Importance:")
print(importance_df)


Feature Importance:
        feature  importance
2         month    0.192075
15    pm25_lag1    0.183246
9           u10    0.182289
14        tp_mm    0.092072
11   wind_speed    0.054033
0           lat    0.045585
10          v10    0.043746
7         skt_c    0.037124
13         ssrd    0.034388
8        sp_hpa    0.030339
6         d2m_c    0.030151
1           lon    0.030044
12         sshf    0.019798
3   day_of_week    0.012630
5         t2m_c    0.012481
4    is_weekend    0.000000


## Feature Importance XGBoost

### Interpretasi Feature Importance

- Variabel `u10` dan `v10` memiliki importance tertinggi, menunjukkan bahwa komponen arah dan pergerakan angin berperan besar dalam memengaruhi distribusi PM2.5.

- Fitur `pm25_lag1` juga memberikan kontribusi besar, menandakan bahwa konsentrasi PM2.5 pada waktu sebelumnya sangat memengaruhi kondisi PM2.5 saat ini. Hal ini menunjukkan adanya dependensi temporal yang kuat.

- Variabel polutan lain seperti `pm10`, `o3`, `so2`, `co`, dan `no2` turut berkontribusi dalam model karena memiliki keterkaitan dengan sumber emisi dan kualitas udara.

- Variabel meteorologi seperti:
  - `tp_mm` (curah hujan),
  - `d2m_c` (dew point),
  - `sp_hpa` (surface pressure),
  - `wind_speed`,
  - `skt_c`,
  - dan `ssrd`
  
  membantu model menangkap pengaruh kondisi atmosfer terhadap konsentrasi PM2.5.

- Fitur `month` memiliki importance yang cukup tinggi, menunjukkan adanya pola musiman pada data PM2.5.

- Variabel spasial `lon` masih memberikan kontribusi terhadap model, sedangkan `lat` memiliki importance sangat rendah sehingga pengaruh spasial linear sederhana belum terlalu dominan pada model XGBoost.

- Variabel kalender seperti `day_of_week` dan `is_weekend` memiliki importance mendekati nol, menunjukkan bahwa pola mingguan tidak terlalu berpengaruh terhadap variasi PM2.5 pada data yang digunakan.

---

### Kesimpulan

- Model XGBoost lebih banyak dipengaruhi oleh:
  - fitur temporal,
  - dinamika angin,
  - dan variabel meteorologi.

- Hasil ini menunjukkan bahwa variasi PM2.5 pada model lebih didorong oleh kondisi atmosfer dan ketergantungan waktu dibanding faktor kalender sederhana.

In [10]:
train_df['pred_xgb'] = train_pred
test_df['pred_xgb']  = test_pred

train_df['residual'] = (
    train_df['pm25']
    - train_df['pred_xgb']
)

test_df['residual'] = (
    test_df['pm25']
    - test_df['pred_xgb']
)

In [11]:
print("\nResidual Summary:")
print(test_df['residual'].describe())


Residual Summary:
count    755.000000
mean      -5.602232
std       15.643288
min      -71.547523
25%      -15.923248
50%       -5.800419
75%        3.456135
max       46.267643
Name: residual, dtype: float64


# Save Residual Dataframe

In [12]:
train_df['dataset_type'] = 'train'
test_df['dataset_type'] = 'test'

residual_df = pd.concat([
    train_df,
    test_df
], axis=0)

residual_df = residual_df.sort_values(
    ['stasiun', 'tanggal']
)

output_path = "outputs/xgb_residuals_full.csv"

residual_df.to_csv(
    output_path,
    index=False
)

print(f"Saved to: {output_path}")

print(residual_df.shape)

Saved to: outputs/xgb_residuals_full.csv
(3950, 30)


## Penyimpanan Residual untuk Analisis Lanjutan

- Setelah model XGBoost selesai dibangun dan dievaluasi, residual dihitung sebagai selisih antara nilai aktual PM2.5 dan hasil prediksi model.

- Residual kemudian disimpan ke dalam dataframe untuk digunakan pada tahap analisis berikutnya.

- Penyimpanan residual bertujuan untuk:
  - mengevaluasi pola kesalahan model,
  - mengidentifikasi kemungkinan autokorelasi,
  - mendeteksi heteroskedastisitas,
  - serta melihat kemungkinan adanya pola spasial yang belum sepenuhnya dijelaskan oleh model XGBoost.

- Tahap ini penting karena residual yang masih memiliki pola tertentu dapat mengindikasikan bahwa masih terdapat struktur data yang belum terakomodasi dalam model.

- Analisis residual akan dilanjutkan pada tahap diagnostik residual dan pemodelan spasial Bayesian untuk mengevaluasi efek spasial yang tersisa (*remaining spatial effects*).